# L12a: Synthetic Markets with Jump Hidden Markov Models
In this lecture, we build synthetic markets: simulated price histories that share the features of real ones, such as fat tails and volatility clustering. A backtest tests a trading method on one historical path. How would the same method do on thousands of plausible paths, including sell-offs that never happened?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
> * __Construct Markov chains and hidden Markov models:__ Define a state space and a transition matrix, compute the stationary distribution, and describe how hidden states emit the values we observe.
> * __Build a jump hidden Markov model of one asset:__ Bin the daily growth rate into states, estimate the transition matrix by counting, fit an emission distribution in each state, and explain why jump episodes are needed to reproduce volatility clustering.
> * __Compose a synthetic market of many stocks:__ Combine a market draw, a sector draw, and a stock's own draw into synthetic stocks that move together like real ones.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
> * A Markov chain is a set of states $\mathcal{S}$ and a transition matrix $\mathbf{P}$ whose rows sum to one. Its stationary distribution satisfies $\bar{\pi} = \bar{\pi}\mathbf{P}$.
> * The jump hidden Markov model bins the daily growth rate into states of equal probability, estimates $\mathbf{P}$ by counting, and draws each day's growth rate from a Student-t distribution of the day's state. Jump episodes, which start with probability $\epsilon$ and last $\lambda$ days on average, keep volatility clustering.
> * A synthetic stock is $g_{i} = \alpha_{i} + \beta_{i}\,g_{m} + \delta_{i}\,f_{s} + s_{i}\,\tilde{\varepsilon}_{i}$: the market and sector draws it shares with other stocks make them move together.
>
> __For understanding__ (you will not need to reproduce or memorize these):
> * The power iteration algorithm, the tuning objective for $\epsilon$ and $\lambda$, and how the sector loadings and the own-draw scale are chosen.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L12a) lists this lecture's symbols with their meanings and units.

Let's get started!
___

## Examples
__We will work through these examples in lecture:__

> [▶ Build a jump hidden Markov model of `SPY`](CHEME-5660-L12a-Example-JumpHMM-SPY-Fall-2026.ipynb). In this example, we fit a hidden Markov model to the daily growth rate of `SPY` from 2014 to 2024, simulate it, and compare its distribution and volatility clustering with the data. We then tune jump episodes and see how much of the clustering they restore.

> [▶ Build a synthetic market of many stocks](CHEME-5660-L12a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model, compare the correlations, volatilities, and β of a synthetic path with the real data, let β drift, and save the generator for later use.

___

## Why Synthetic Markets?
Previously, we tested trading methods by backtesting them on historical prices. A backtest is a single path: the market went through one sequence of booms and sell-offs, and a method that did well on that sequence may have been lucky. To judge a method, we would like to run it on many plausible paths, including bad ones that did not happen.

A __synthetic market__ is a model that generates such paths. To be useful for stress tests, its paths should share the features of real growth rates that matter for risk:
* __Fat tails:__ large daily moves are far more common than a normal distribution predicts.
* __Volatility clustering:__ large moves follow large moves, so the absolute growth rate stays correlated for weeks.
* __Co-movement:__ stocks move together, through the market and through their sectors.

Geometric Brownian motion, which we used before, has none of the three. In this lecture, we build a generator that has all three, starting from Markov chains.
___

## Markov Chains
A discrete Markov model is a stochastic (random) process that describes the transitions between states based on defined probabilities. Formally, a Markov model is determined by the tuple $\mathcal{M} = (\mathcal{S},\mathbf{P})$, where $\mathcal{S}$ represents the set of states in the model and $\mathbf{P}$ is the transition matrix.

### The state space $\mathcal{S}$
The state space $\mathcal{S}$ is the set of all possible values a system can assume. For example, if a Markov chain can be in the states $\left\{1,2,3\right\}$, then $\mathcal{S} \equiv \left\{1,2,3\right\}$. What do these states represent? The answer depends on the problem. Here are two concrete examples:
* __Letters in words__ $\mathcal{S} \equiv \left\{a,b,c,\dotsc,z\right\}$: If the state space $\mathcal{S}$ is defined as the alphabet, we could develop a Markov model to generate $n$ words, each of length $l$ characters, that start with a specific letter such as $t$.
* __Investor moods__ $\mathcal{S} \equiv \left\{\text{bullish},\text{neutral},\text{bearish}\right\}$: In this case, the state space is defined as three possible moods an investor can experience. Using this type of $\mathcal{S}$, we could build a Markov model to simulate how an investor's mood changes as they watch the market, the news, and other factors.

Next, let's look at the Markov property and the transition matrix $\mathbf{P}$.

### The Markov Property
The Markov property is the key characteristic of Markov models: a random process has this property when the probability of future states depends only on the present state, not on how it got there. This memoryless property simplifies how we model stochastic systems. From weather prediction to inventory management, the Markov property captures the behavior of many real-world systems.

> __Definition:__ A stochastic process $\{X_n\}_{n=0}^{\infty}$ satisfies the **Markov property** if, for all $n \geq 0$ and all sequences of states $s_0, s_1, \ldots, s_n, s_{n+1}$ in the state space $\mathcal{S}$:
>
> $$P(X_{n+1} = s_{n+1} \mid X_0 = s_0, X_1 = s_1, \ldots, X_n = s_n) = P(X_{n+1} = s_{n+1} \mid X_n = s_n)$$
>
> In other words, given the present state $X_n = s_n$, the future state $X_{n+1} = s_{n+1}$ is conditionally independent of all past states $X_0 = s_0, X_1 = s_1, \ldots, X_{n-1} = s_{n-1}$. This conditional independence captures the essence of memorylessness: the past is irrelevant once we know the present.

#### One-Step Transition Probabilities

The one-step transition probability describes how the system moves from one state to another in a single time step. For a Markov chain with finite state space $\mathcal{S} = \{s_1, s_2, \ldots, s_N\}$, we define:

$$
\begin{equation*}
p_{ij} = P(X_{n+1} = s_j \mid X_n = s_i)
\end{equation*}
$$

Here, $p_{ij}$ is the probability of moving from state $s_i$ at time $n$ to state $s_j$ at time $n+1$. Because of the Markov property, this probability does not depend on $n$ (as long as the process is time-homogeneous, meaning transition probabilities stay constant over time). We pack these probabilities into the transition matrix $\mathbf{P}$.

#### Why the Markov Property Matters
The Markov property simplifies our modeling work: instead of tracking the entire history of a process, we only need to know the present state. Without this property, the number of possible historical sequences would grow exponentially, making analysis difficult. With it, the problem becomes tractable and easier to work with.

> __Why is this cool?__ The Markov property lets us build probability models of complex systems, use dynamic programming algorithms to make optimal decisions (such as value iteration for MDPs), and analyze long-run behavior using matrix operations and eigenvalue analysis. This approach has proven useful in finance, operations research, biology, physics, and machine learning.

The Markov property is an assumption about the real world, and not all stochastic processes satisfy it exactly. However, by choosing the state representation carefully to capture the relevant information, we can often make the Markov assumption work well in practice.

### The Transition Matrix
A discrete Markov chain is a sequence of random variables (states) $X_{1},\dotsc, X_{n}$ with the _Markov property_ (i.e., the probability of moving to the next state depends only on the present state and not on past states):
$$
\begin{equation*}
P(X_{n+1} = s_{n+1} \mid X_{1}=s_{1}, \dots, X_{n}=s_{n}) = P(X_{n+1} = s_{n+1} \mid X_{n} = s_{n})
\end{equation*}
$$
For finite state spaces $\mathcal{S}$, the probability of moving from state $s_{i}\rightarrow{s_{j}}$ in the next step $n\rightarrow{n+1}$ is encoded in the transition matrix $p_{ij}\in\mathbf{P}\in\mathbb{R}^{|\mathcal{S}|\times|\mathcal{S}|}$: 
$$
\begin{equation*}
p_{ij} = P(X_{n+1}~=~s_{j}~\mid~X_{n}~=~s_{i})
\end{equation*}
$$

The transition matrix $\mathbf{P}$ represents the state transitions of a Markov chain:

<div>
    <center>
        <img src="figs/Fig-TransitionMatrix-Schematic.svg" width="680"/>
    </center>
</div>

The transition matrix $\mathbf{P}\in\mathbb{R}^{|\mathcal{S}|\times|\mathcal{S}|}$ has the following properties:
* All elements of the transition matrix $\mathbf{P}$ are non-negative: $p_{ij}\geq{0}$.  
* The rows of $\mathbf{P}$ represent the current states, while the columns represent the future states (our convention). Thus, the element $p_{ij}$ in row $i$ and column $j$ represents the probability of transitioning from state $s_{i}$ to state $s_{j}$ in the next step.
* The rows of $\mathbf{P}$ must sum to unity (i.e., each row encodes the probability of all possible future outcomes given where we are currently). Thus, for any row $i$, we have: $\sum_{j=1}^{|\mathcal{S}|} p_{ij} = 1$. This means that the transition probabilities from state $s_{i}$ to all other states sum to 1 (we have a fixed number of states, and we must transition to one of them).
* If the transition matrix $\mathbf{P}$ is invariant, then $p_{ij}$ does not change as $n\rightarrow{n+1}~\forall{n}$. In other words, the probability of transitioning from state $i$ to state $j$ does not change as the system evolves. The $p_{ij}$ values are constant.

> __Example:__ Consider a simple 2-state Markov model for weather with states $\mathcal{S} = \{\text{Sunny}, \text{Rainy}\}$. The transition matrix is:
>
> $$\mathbf{P} = \begin{bmatrix} 0.8 & 0.2 \\ 0.4 & 0.6 \end{bmatrix}$$
>
> where row 1 corresponds to Sunny and row 2 corresponds to Rainy. If today is Sunny (state 1), there is an 80% chance tomorrow is Sunny and a 20% chance it is Rainy. If today is Rainy (state 2), there is a 40% chance tomorrow is Sunny and a 60% chance it is Rainy. Starting with the initial distribution $\pi_0 = [1, 0]$ (certain it is Sunny today), after one day we have:
>
> $$\pi_1 = \pi_0 \mathbf{P} = [1, 0]\begin{bmatrix} 0.8 & 0.2 \\ 0.4 & 0.6 \end{bmatrix} = [0.8, 0.2]$$
>
> This means there is an 80% probability of being in the Sunny state and a 20% probability of being in the Rainy state tomorrow.

#### Estimating the transition matrix from data
In finance, we do not know $\mathbf{P}$; we estimate it from a sequence of observed states $X_{1},\dotsc,X_{T}$. Let $n_{ij}$ be the number of days in state $s_{i}$ followed by a day in state $s_{j}$. The estimate of $p_{ij}$ is the share of the days in state $s_{i}$ that moved to state $s_{j}$:
$$
\hat{p}_{ij} = \frac{n_{ij}}{\sum_{k=1}^{|\mathcal{S}|} n_{ik}}
$$
Each row of $\hat{\mathbf{P}}$ sums to one by construction. This counting estimate is the maximum likelihood estimate of $\mathbf{P}$, and it is how we will estimate the transition matrix of a market generator.

### The Stationary Distribution
The stationary distribution $\bar{\pi}$ of a Markov chain with transition matrix $\mathbf{P}$ is a probability distribution that remains unchanged as the system evolves: $\bar{\pi} = \bar{\pi}\mathbf{P}$. It is the long-run share of time the chain spends in each state, and it is where a generator starts its paths. We estimate $\bar{\pi}$ by repeatedly applying the transition matrix to an initial distribution until convergence:

$$
\begin{align*}
\pi_1 &= \pi_0 \mathbf{P} \\
\pi_2 &= \pi_1 \mathbf{P} = \pi_0 \mathbf{P}^2 \\
\pi_3 &= \pi_2 \mathbf{P} = \pi_0 \mathbf{P}^3 \\
&\vdots \\
\pi_k &= \pi_{k-1} \mathbf{P} = \pi_0 \mathbf{P}^k
\end{align*}
$$

__Initialize__: Given an ergodic transition matrix $\mathbf{P}$ with $N$ states, tolerance $\text{tol}$, and maximum number of iterations $T$, initialize the iteration counter $t\gets 1$, the uniform initial distribution $\pi \gets \left[1/N,\ldots,1/N\right]$, and $\texttt{converged}\gets\texttt{false}$.

While $\texttt{converged}$ is $\texttt{false}$ __do__:
1. Compute: $\pi^{\prime} \gets \pi\mathbf{P}$.
2. Check for convergence:
    - If $\lVert \pi^{\prime} - \pi \rVert_{1} \leq \text{tol}$, then set $\texttt{converged}\gets\texttt{true}$ and $\bar{\pi}\gets\pi^{\prime}$.
    - If $\lVert \pi^{\prime} - \pi \rVert_{1} > \text{tol}$, increment $t\gets{t+1}$. Set $\pi\gets\pi^{\prime}$ and continue.
3. Check for maximum iterations reached:
    - If $t\geq{T}$, then set $\texttt{converged}\gets\texttt{true}$ and return $\bar{\pi}\gets\pi^{\prime}$. Notify that maximum iterations were reached without convergence.

__Output__: The stationary distribution $\bar{\pi}$.

The convergence rate depends on the spectral gap (difference between the largest and second-largest eigenvalue magnitudes of $\mathbf{P}$), with larger gaps enabling faster convergence.

__Notes:__
1. This algorithm assumes $\mathbf{P}$ is ergodic (irreducible and aperiodic), ensuring a unique stationary distribution exists.
2. The $\pi^{\prime} \gets \pi\mathbf{P}$ step is a left vector-matrix multiplication: $\pi$ is a row vector, and the new probability of state $s_{j}$ is $\pi^{\prime}_{j} = \sum_{i}\pi_{i}p_{ij}$.
3. The stationary distribution satisfies the equation $\bar{\pi} = \bar{\pi}\mathbf{P}$, meaning it remains unchanged under the transition matrix.
___

## Hidden Markov Models (HMMs)
Next, let's explore hidden Markov models (HMMs), which extend Markov models with hidden states that we do not observe directly.


> __Definition:__
>
> A discrete hidden Markov model (HMM) is similar to a discrete observable Markov model, but with an added layer of complexity: the states are not directly observable. Instead, we observe outputs (or emissions) that are probabilistically related to the hidden states. An HMM is characterized by the tuple $\mathcal{M} = (\mathcal{S},\mathcal{O},\mathbf{P},\mathbf{E})$:
> * __State Space ($\mathcal{S}$)__: The set of the $|\mathcal{S}|$ hidden states that the system can be in.
> * __Observation Space ($\mathcal{O}$)__: The set of the $|\mathcal{O}|$ observable states (emissions) that can be observed.
> * __Transition Matrix ($\mathbf{P}$)__: The transition matrix $\mathbf{P}\in\mathbb{R}^{|\mathcal{S}|\times|\mathcal{S}|}$ that defines the probabilities of transitioning between hidden states.
> * __Emission Matrix ($\mathbf{E}$)__: The emission matrix $\mathbf{E}\in\mathbb{R}^{|\mathcal{S}|\times|\mathcal{O}|}$ that defines the probabilities of observing each observable state given a hidden state.

The transition and emission matrices have the following (summation) properties:

* The transition matrix $\mathbf{P}$ encodes the probability of moving from one hidden state to 
another hidden state at the next time step. For state indices $i,j \in \{1,\ldots,|\mathcal{S}|\}$, the element $p_{ij} = P(X_{n+1} = s_j | X_{n} = s_i)$ must satisfy:
    $$
    \begin{align*}
    \sum_{j=1}^{|\mathcal{S}|} p_{ij} &= 1\qquad\forall{i\in\{1,\ldots,|\mathcal{S}|\}}
    \end{align*}
    $$
* The emission matrix $\mathbf{E}$ encodes the probability of emitting an observable state from a hidden state. For hidden state index $i \in \{1,\ldots,|\mathcal{S}|\}$ and observation index $j \in \{1,\ldots,|\mathcal{O}|\}$, the element $e_{ij} = P(O_{n} = o_j\,| X_{n} = s_i)$ must satisfy:
    $$
    \begin{align*}
    \sum_{j=1}^{|\mathcal{O}|} e_{ij} &= 1\qquad\forall{i\in\{1,\ldots,|\mathcal{S}|\}}
    \end{align*}
    $$

### Sampling Algorithm
Let's outline a simple sampling algorithm for generating random sequences from an HMM.

__Initialize__: Given an HMM defined by $\mathcal{M} = (\mathcal{S},\mathcal{O},\mathbf{P},\mathbf{E})$, the number of time steps $T$, and the stationary distribution $\bar{\pi}$. Draw an initial hidden state $X_0$ from the initial distribution: $X_0\sim \bar{\pi}$.

For $n = 1$ to $T$ __do__:
1. Sample the next __hidden state__ $X_n$ from the $\texttt{categorical}$ distribution $\mathbb{P}_{H}(...)$ defined by the previous state $X_{n-1}$ and the transition matrix $\mathbf{P}$: $X_n \sim \mathbb{P}_{H}(X_n \mid X_{n-1})$
2. Sample the observable state $O_n$ from the $\texttt{categorical}$  distribution $\mathbb{P}_{E}(...)$ defined by the current hidden state $X_n$ and the emission matrix $\mathbf{E}$: $O_n \sim \mathbb{P}_{E}(O_n \mid X_n)$
3. Store $X_n$ and $O_n$.

__Output__: The sequences of hidden states $\{X_1, X_2, \ldots, X_T\}$ and observable states $\{O_1, O_2, \ldots, O_T\}$. The initial state $X_0$ is used to generate the sequence but is not included in the output.

> __What is a categorical distribution?__ A categorical distribution is a probability distribution that describes the possible outcomes of a random variable that can take on one of $k$ different categories, each with its own probability. It is a generalization of the Bernoulli distribution (which has two categories) to multiple categories. The probabilities for each category must sum to 1.

So how do we do this? We create categorical distributions from the rows of the transition and emission matrices, and then sample from these distributions at each time step to generate the sequences of hidden and observable states.

#### Continuous emissions
For a market generator, the observation is a growth rate, which is a number, not one of a few categories. So instead of an emission matrix $\mathbf{E}$, each hidden state $s_{k}$ has an emission __distribution__, a probability density of the growth rate given the state. The sampling algorithm is the same, except that step 2 draws the growth rate from the emission distribution of the current state:
$$
g_{n} \mid X_{n} = s_{k} \;\sim\; \text{emission distribution of } s_{k}
$$
The hidden state carries the information about how volatile the market is today; the emission turns it into a growth rate.
___

## The Jump Hidden Markov Model
Let's build a hidden Markov model of the daily growth rate $g_{t} = \frac{1}{\Delta t}\ln\left(S_{t}/S_{t-1}\right)$ (units: 1/yr) of one asset, following Alswaidan and Varner, who call it a jump hidden Markov model. It has three parts:
* __States:__ We fit a Laplace distribution, with location $\mu$ and scale $b$, to the growth rates and cut it into $N$ bins of equal probability. The bin edges are the quantiles $F_{L}^{-1}(k/N)$ for $k = 1,\dotsc,N-1$, where $F_{L}$ is the Laplace cumulative distribution function. Each bin is a hidden state; state 1 holds the most negative growth rates and state $N$ the most positive. Each day of the data falls in one bin, which gives the sequence of states we count transitions from. In the data, the states are observed this way; in simulation, they are hidden, and only the growth rates they emit are reported.
* __Transition matrix:__ We estimate $\hat{\mathbf{P}}$ by counting the moves between states on consecutive days.
* __Emissions:__ In each state $s_{k}$, the growth rate is drawn from a Student-t distribution, located and scaled by the growth rates of the days in that state: $g_{t} \mid X_{t} = s_{k} \sim \mu_{k} + \sigma_{k}\,T_{\nu}$, where $T_{\nu}$ is a Student-t random variable with $\nu$ degrees of freedom.

The Student-t emissions and the wide extreme bins give the generator fat tails. To simulate a path, we draw a starting state from $\bar{\pi}$, step the chain with $\hat{\mathbf{P}}$, and draw each day's growth rate from the emission of the day's state.

### Why add jumps?
The counted chain forgets quickly. A volatile day puts the chain in an extreme state, but the next few transitions usually take it back to the middle states, so the absolute growth rate of a simulated path is correlated for only a few days. In real markets, volatile periods last for weeks.

The jump hidden Markov model adds __jump episodes__ to keep volatile periods going:
* On a day with no episode under way, an episode starts with probability $\epsilon$.
* The episode lasts a Poisson number of days with mean $\lambda$.
* On each episode day, the state is drawn from the few most extreme states: the most negative ones with a probability slightly above one half, the most positive ones otherwise. An episode is a run of large moves of both signs, not a single price jump.

### Tuning and checking the model
We choose $\epsilon$ and $\lambda$ by a grid search. For each pair, we simulate paths, keep the paths that contain an episode, and score them by how far their autocorrelation of $|g_{t}|$ at lags 1 to 25 and their kurtosis are from the data's:
$$
J(\epsilon,\lambda) = \sum_{k=1}^{25}\left(\rho_{k} - \hat{\rho}_{k}\right)^{2} + 0.2\left(\kappa - \hat{\kappa}\right)^{2}
$$
where $\rho_{k}$ is the autocorrelation of $|g_{t}|$ at lag $k$ and $\kappa$ the kurtosis of the data, and $\hat{\rho}_{k}$, $\hat{\kappa}$ those of a simulated path, averaged over the paths. We keep the pair with the smallest $J$. Finally, we check the tuned model against the data: the distribution of each simulated path (a Kolmogorov–Smirnov test), its kurtosis, and its volatility clustering.

> __Example:__
>
> [▶ Build a jump hidden Markov model of `SPY`](CHEME-5660-L12a-Example-JumpHMM-SPY-Fall-2026.ipynb). In this example, we fit a hidden Markov model to the daily growth rate of `SPY` from 2014 to 2024, simulate it, and compare its distribution and volatility clustering with the data. We then tune jump episodes and see how much of the clustering they restore.

Next, let's extend the generator from one asset to many.
___

## Many Assets: The Sector Factor Model
A stress test of a portfolio needs synthetic stocks that move together the way real ones do. We could fit a jump hidden Markov model to every stock and draw them independently, but then they would not move together at all. The __sector factor model__ builds each synthetic stock from three draws: a market draw and a sector draw that it shares with other stocks, and a draw of its own. The daily growth rate of stock $i$ in sector $s$ is:
$$
g_{i}(t) = \alpha_{i} + \beta_{i}\,g_{m}(t) + \delta_{i}\,f_{s}(t) + s_{i}\,\tilde{\varepsilon}_{i}(t)
$$
where:
* $g_{m}(t)$ is a draw of a jump hidden Markov model of the market (`SPY`), and $\alpha_{i}$ and $\beta_{i}$ are the stock's single index model estimates, which we hold fixed.
* $f_{s}(t)$ is a draw of a jump hidden Markov model of sector $s$'s factor: the average single index residual of the stocks in the sector. The loading $\delta_{i}$ is the slope of the stock's residual on the other residuals in its sector, scaled so the sector's co-movement matches the data.
* $\tilde{\varepsilon}_{i}(t)$ is a draw of a jump hidden Markov model of the stock itself, centered on its long-run mean. Its scale $s_{i}$ is chosen so the synthetic stock keeps its real variance: $s_{i}^{2}$ is the share of the stock's variance that the market and sector terms leave unexplained.

On each day, every stock in a sector uses the same market draw and the same sector draw, which is why they move together.

### What the model keeps and what it gives up
Each synthetic stock keeps its own fat tails and volatility clustering, because every draw comes from a jump hidden Markov model. The stocks in a sector move together through their shared sector draw, and all stocks move together through the market draw. The sector draws are independent of each other, so two stocks in different sectors are linked only through the market, and the synthetic correlations across sectors come out a little lower than the real ones.

Why not a copula? A copula imposes a fitted correlation by reordering each stock's draws across days, which would scramble the timing of its volatile periods and its jump episodes. Composing stocks from shared draws on the same day keeps each stock's dynamics intact.

### An option: a drifting β
Holding $\beta_{i}$ fixed is a simplification: real β wanders over time. The model can let $\beta_{i}(t)$ drift slowly around $\hat{\beta}_{i}$, as an exponential moving average of random shocks, sized by how much the online estimate of β wanders in the real data. A trading method that re-estimates β as it trades then faces the same problem in the synthetic market that it faces in the real one.

> __Example:__
>
> [▶ Build a synthetic market of many stocks](CHEME-5660-L12a-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model, compare the correlations, volatilities, and β of a synthetic path with the real data, let β drift, and save the generator for later use.
___

## Summary
This lecture builds synthetic markets for stress tests, from Markov chains and hidden Markov models to a jump hidden Markov model of one asset and a sector factor model of many stocks.

> __Key Takeaways:__
>
> * __Markov chains and hidden Markov models describe sequences with memory of the present state:__ We defined the state space, the transition matrix, and the stationary distribution, estimated the transition matrix by counting transitions, and extended the chain with hidden states whose emission distributions produce the values we observe.
> * __The jump hidden Markov model reproduces fat tails and part of volatility clustering:__ We binned the daily growth rate into states of equal probability, counted transitions, and drew growth rates from a Student-t distribution in each state. Because the counted chain forgets volatile days quickly, we added jump episodes and tuned their probability and length against the data.
> * __Shared draws make synthetic stocks move together:__ We composed each synthetic stock from a market draw and a sector draw that it shares with other stocks, plus a draw of its own, keeping each stock's own dynamics and its real variance.

Synthetic markets let us test a trading method on many plausible paths, including sell-offs that never happened, rather than on one historical path.
___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___